# HUÁNUCO ECOGUARD AI · Demostración académica
Este notebook **no entrena**. Restaura `best.pt` de la entrega original de Drive y analiza una fotografía por solicitud.
Selecciona **Entorno de ejecución → Cambiar tipo de entorno → GPU T4**.
Carga el paquete `dist/huanuco-ecoguard-*.zip` construido en Linux. Los pesos permanecen respaldados en Drive.
Las celdas no contienen resultados inventados ni salidas precargadas.


In [ ]:
from google.colab import drive, files, output
drive.mount('/content/drive')


Carga el paquete académico ZIP generado por `python3 ecoguard.py empaquetar`, no el respaldo del dataset.

In [ ]:
from pathlib import Path
import zipfile, sys, hashlib, json
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Carga un solo paquete académico ZIP.')
archive = Path(next(iter(uploaded)))
print('SHA256 del paquete:', hashlib.sha256(archive.read_bytes()).hexdigest())
# Compara este SHA256 con el mostrado por ecoguard.py empaquetar en Linux.
with zipfile.ZipFile(archive) as z:
    # Leer únicamente la biblioteca estándar de verificación del paquete propio.
    code = z.read('entrega/scripts/artefactos.py').decode('utf-8')
namespace = {'__name__': 'ecoguard_integrity'}
exec(compile(code, 'artefactos.py', 'exec'), namespace)
namespace['extract_original'](archive, '/content/ecoguard-academico')
ROOT = Path('/content/ecoguard-academico/entrega')
print('Archivos con SHA256 correcto:', namespace['verify_manifest'](ROOT))
sys.path.insert(0, str(ROOT / 'scripts'))


In [ ]:
import subprocess
# Solo instala dependencias en Colab; usa PyTorch/CUDA del runtime.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(ROOT / 'requirements.txt')], check=True)


In [ ]:
from demo_colab import prepare
original = prepare()
weights = original / 'train100/weights/best.pt'
print('Pesos originales verificados:', weights)


## Pruebas reales en Colab
Carga el modelo, comprueba las clases, analiza una fotografía original, rechaza archivos inválidos y verifica Gradio mediante HTTP/API. No repite entrenamiento.

In [ ]:
import os
env = {**os.environ, 'ECOGUARD_REAL_TESTS': '1', 'ECOGUARD_WEIGHTS': str(weights)}
subprocess.run([sys.executable, str(ROOT / 'tests/test_app_real.py')], env=env, check=True)


## Aplicación privada dentro de Colab
Usa el proxy autenticado de tu notebook. No requiere publicar un sitio permanente ni depender de gradio.live. Mantén este runtime abierto durante la exposición.

In [ ]:
from app_gradio import build_app
app, _ = build_app(weights)
proxy = output.eval_js('google.colab.kernel.proxyPort(7861)')
app.launch(share=False, server_name='0.0.0.0', server_port=7861,
           root_path=proxy, inline=False, prevent_thread_lock=True,
           show_error=False, max_file_size='20mb')
output.serve_kernel_port_as_iframe(7861, height=950)


Si el navegador bloquea el iframe, abre el proxy autenticado mostrado abajo. La aplicación puede reconstruirse desde este notebook y Drive aunque caduque el runtime.
Para un enlace temporal opcional, detén esta app y ejecuta `python3 ecoguard.py app` desde WSL con la sesión CLI autenticada. No ejecutes los comandos del entrenador.


In [ ]:
print('Acceso autenticado de esta sesión:', proxy)


In [ ]:
# Ejecutar únicamente al terminar la demostración:
# app.close()
